## ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Project 3: NLP Classification: Subreddit Pepsi vs Coca-Cola | Part 2: Vectorizer

---

[README](../README.md) | [Part 1: EDA](01_EDA.ipynb) | **Part 2: Vectorizer** | [Part 3: xxx](03_Interpretation.ipynb)

---

### Introduction
From [Part 1: EDA](01_EDA.ipynb), we obtained `subreddit_pepsi_vs_cocacola.csv_clean`, which has been cleaned. In this part, we focus on tuning the hyperparameters of two vectorizers: **CountVectorizer** and **TfidfVectorizer**, while using default settings for the model. Our goal is to analyze how the hyperparameters of the vectorizers affect model performance.

The **CountVectorizer** represents each document by the frequency of words (tokens) in the text. While **TfidfVectorizer** measures the relative importance of a word in a document by balancing its frequency in that document (Term Frequency) with how rare it is across all documents (Inverse Document Frequency). The latter vectorizer often provides better results, especially when distinguishing key terms in large corpora.

### Import

In [5]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP tools
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Model selection and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, confusion_matrix, recall_score, precision_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classifiers
from sklearn.naive_bayes import MultinomialNB 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Time utility
import time
from joblib import parallel_backend

# Set seed for reproducibility
np.random.seed(42)             

### Data Preparation

In [7]:
df = pd.read_csv('../data/subreddit_pepsi_vs_cocacola_clean.csv')          # Load Data
df.head(1)                                                                 # Check first row

,title,score,id,url,comms_num,created,body,is_pepsi,title_body,title_body_length,title_body_word_count
0,Wall clock.,17,1godhom,https://www.reddit.com/gallery/1godhom,1,11/11/2024 5:58,I'm trying to locate a value for this clock. I...,0,Wall clock. I'm trying to locate a value for t...,220,39


In [8]:
df.shape

(1958, 11)

In [9]:
# Set Features and Target

X = df['title_body']
y = df['is_pepsi']

# Split the data to training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X
                                                    , y
                                                    , test_size = 0.2
                                                    , stratify = y
                                                   )

### Vectorizer
- We focus on two types of vectorizers: **CountVectorizer** and **TfidfVectorizer**, which share the same set of parameters.
- First, converting the text to lowercase and removing words related to **Pepsi** and **Coca-Cola** are mandatory steps.
- The other parameters we will adjust for comparison are:
    - Removes English Stopwords or not
    - Maximum features: 3000, 5000, or None
    - N-gram length: whether to include bigrams or not
    - Minimum document frequency (min_df): 2, or 3
    - Maximum document frequency (max_df): 0.8 or 0.9

In [11]:
# Related words are words that can easily identify the subreddit categories, 
# such as the names of brands.
related_words = {'pepsi', 'pepsico', 'coca', 'cola', 'coke'}

# Custom stop words are common English words that are meaningless, 
# and we also include related words to create a custom stop word list.

custom_stop_words = set(stopwords.words('english')).union(related_words)

In [12]:
# Instantiate Stemmer and Lemmatizer
stemmer = nltk.PorterStemmer()
lemmatizer = nltk.WordNetLemmatizer()

# Create 6 custom tokenizers. 
# Note that all tokenizers are designed to remove symbols, related words, and convert text to lowercase, 
# with the following additional functions:
# - lower_only: no other changes
# - lower_stop: removes stopwords
# - stem_only: applies stemming
# - stem_stop: applies stemming and removes stopwords
# - lem_only: applies lemmatizing
# - lem_stop: applies lemmatizing and removes stopwords

def lower_only(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [token for token in tokens if token not in related_words and token.isalpha()]

def lower_stop(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [token for token in tokens if token not in custom_stop_words and token.isalpha()]

def stem_only(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [stemmer.stem(token) for token in tokens if token not in related_words and token.isalpha()]

def stem_stop(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [stemmer.stem(token) for token in tokens if token not in custom_stop_words and token.isalpha()]

def lem_only(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [lemmatizer.lemmatize(token) for token in tokens if token not in related_words and token.isalpha()]

def lem_stop(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [lemmatizer.lemmatize(token) for token in tokens if token not in custom_stop_words and token.isalpha()]

In [13]:
# Custom metrics used in GridSearch results
scorers = {
    'accuracy': make_scorer(accuracy_score)
    , 'recall': make_scorer(recall_score, average = 'binary')
    , 'precision': make_scorer(precision_score, average = 'binary')
    , 'f1': make_scorer(f1_score, average = 'binary')
}

In [14]:
# GridSearch parameters
# 'vectorizer__' is used to set parameters for CountVectorizer and TfidfVectorizer.
# Since the tokenizer parameter is applied, we set 'token_pattern' to None.
params = {
    'vectorizer': [CountVectorizer(token_pattern = None)
                   , TfidfVectorizer(token_pattern = None)
                  ] 
    , 'vectorizer__tokenizer': [lower_only
                                , lower_stop
                                , stem_only
                                , stem_stop
                                , lem_only
                                , lem_stop
                               ] 
    , 'vectorizer__max_features': [3000
                                   , 5000
                                   , None
                                  ]
    , 'vectorizer__ngram_range': [(1, 1)                  # unigrams
                                  , (1, 2)                # unigrams and bigrams
                                 ]
    , 'vectorizer__min_df': [2, 3]
    , 'vectorizer__max_df': [0.8, 0.9]
    , 'classifier': [#MultinomialNB()
                     #, LogisticRegression()
                     #, KNeighborsClassifier()
                     #, DecisionTreeClassifier()
                     #, BaggingClassifier()
                     #, RandomForestClassifier()
                      AdaBoostClassifier(algorithm = 'SAMME')
                     #, GradientBoostingClassifier()
                     #, SVC()
                     #, XGBClassifier()
                    ]
}

In [15]:
# Create Pipeline
pipeline = Pipeline([
    ('vectorizer', CountVectorizer())                    # Will replace in GridSearch  
    , ('classifier', MultinomialNB())                    # Will replace in GridSearch 
])

In [16]:
# GridSearchCV
grid_search = GridSearchCV(pipeline
                           , param_grid=params
                           , cv = 5
                           , verbose = 3
                           , scoring = scorers
                           , refit = 'f1'
                           , return_train_score = True
)

In [17]:
# Fit the grid search

# Record start time
start_time =time.time()



with parallel_backend('threading', n_jobs=-1):
    grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
[CV 3/5] END classifier=AdaBoostClassifier(algorithm='SAMME'), vectorizer=CountVectorizer(token_pattern=None), vectorizer__max_df=0.8, vectorizer__max_features=3000, vectorizer__min_df=2, vectorizer__ngram_range=(1, 1), vectorizer__tokenizer=<function lower_stop at 0x0000022EDAEEE160>; accuracy: (train=0.601, test=0.559) f1: (train=0.718, test=0.695) precision: (train=0.560, test=0.536) recall: (train=0.998, test=0.987) total time=  11.2s
[CV 1/5] END classifier=AdaBoostClassifier(algorithm='SAMME'), vectorizer=CountVectorizer(token_pattern=None), vectorizer__max_df=0.8, vectorizer__max_features=3000, vectorizer__min_df=2, vectorizer__ngram_range=(1, 1), vectorizer__tokenizer=<function lower_stop at 0x0000022EDAEEE160>; accuracy: (train=0.641, test=0.605) f1: (train=0.725, test=0.700) precision: (train=0.594, test=0.571) recall: (train=0.931, test=0.906) total time=  11.8s
[CV 2/5] END classifier=AdaBoostClassifier(algorit

C:\Users\Home\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
3 fits failed out of a total of 1440.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Home\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Home\anaconda3\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Home\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 471, in fit
    Xt = self._fit(X, y, routed_p

In [18]:
# Code you want to time (e.g., fitting a model)
# Example: grid_search.fit(X_train, y_train)

# Record end time
end_time = time.time()

# Calculate elapsed time
elapsed_time = end_time - start_time
print(f"Time taken: {elapsed_time:.4f} seconds")

Time taken: 5061.2015 seconds


In [19]:
# Collect results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results.to_csv('../data/cv_results_ada.csv') 
# Display important metrics
cv_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_vectorizer,param_vectorizer__max_df,param_vectorizer__max_features,param_vectorizer__min_df,param_vectorizer__ngram_range,...,mean_test_f1,std_test_f1,rank_test_f1,split0_train_f1,split1_train_f1,split2_train_f1,split3_train_f1,split4_train_f1,mean_train_f1,std_train_f1
0,10.350171,0.374434,3.880175,0.880687,AdaBoostClassifier(algorithm='SAMME'),CountVectorizer(token_pattern=None),0.8,3000,2,"(1, 1)",...,0.707909,0.011013,210,0.725046,0.720818,0.717833,0.720889,0.730275,0.722972,0.004313
1,12.712624,3.940487,2.590621,1.374750,AdaBoostClassifier(algorithm='SAMME'),CountVectorizer(token_pattern=None),0.8,3000,2,"(1, 1)",...,0.709037,0.010830,186,0.725046,0.720818,0.717833,0.721637,0.722256,0.721518,0.002327
2,18.365387,1.379911,4.883941,0.888284,AdaBoostClassifier(algorithm='SAMME'),CountVectorizer(token_pattern=None),0.8,3000,2,"(1, 1)",...,0.717586,0.015627,25,0.735215,0.721939,0.724743,0.722424,0.733068,0.727478,0.005564
3,20.766616,4.046790,2.685077,1.525852,AdaBoostClassifier(algorithm='SAMME'),CountVectorizer(token_pattern=None),0.8,3000,2,"(1, 1)",...,0.717282,0.015782,37,0.735215,0.721939,0.724743,0.722424,0.728960,0.726656,0.004947
4,15.595309,6.251313,1.673674,2.090103,AdaBoostClassifier(algorithm='SAMME'),CountVectorizer(token_pattern=None),0.8,3000,2,"(1, 1)",...,NaN,NaN,288,NaN,NaN,NaN,0.720385,0.725675,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,13.318210,4.399946,3.760446,3.329728,AdaBoostClassifier(algorithm='SAMME'),TfidfVectorizer(token_pattern=None),0.9,None,3,"(1, 2)",...,0.714200,0.014210,79,0.727273,0.729089,0.730140,0.719340,0.734161,0.728001,0.004884
284,20.417262,1.747585,7.995834,1.998785,AdaBoostClassifier(algorithm='SAMME'),TfidfVectorizer(token_pattern=None),0.9,None,3,"(1, 2)",...,0.709257,0.009391,176,0.721053,0.720805,0.740317,0.726480,0.729380,0.727607,0.007144
285,21.979633,6.704985,3.585828,2.264103,AdaBoostClassifier(algorithm='SAMME'),TfidfVectorizer(token_pattern=None),0.9,None,3,"(1, 2)",...,0.721873,0.018379,13,0.744966,0.729659,0.741544,0.727273,0.732268,0.735142,0.006896
286,17.024531,1.292209,4.146189,0.982747,AdaBoostClassifier(algorithm='SAMME'),TfidfVectorizer(token_pattern=None),0.9,None,3,"(1, 2)",...,0.701348,0.014790,264,0.730745,0.721750,0.730139,0.715051,0.725907,0.724718,0.005821
